# Brain — Multi-Agent System
**Google Colab notebook** — clone the repo, install deps, mount Drive for persistence, run the brain.

Recommended runtime: **L4 GPU** (Colab Pro) or **T4** (free tier).

If you want local Ollama models instead of Gemini, jump to the **Ollama (optional)** section.

## 1 — Mount Google Drive (data persistence)
ChromaDB and the episode SQLite database will be stored on Drive so they survive session restarts.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

# All persistent data goes here — survives session restarts
DRIVE_DATA = '/content/drive/MyDrive/Brain/data'
pathlib.Path(DRIVE_DATA).mkdir(parents=True, exist_ok=True)

# Symlink so the code always finds data/ in the repo root
REPO_DATA = '/content/Brain/data'
if os.path.islink(REPO_DATA):
    os.remove(REPO_DATA)
# (symlink created after clone in step 3)
print(f'Drive mounted. Data will persist at: {DRIVE_DATA}')

Mounted at /content/drive
Drive mounted. Data will persist at: /content/drive/MyDrive/Brain/data


## 2 — Clone the repo

In [2]:
import os

REPO_URL = 'https://github.com/Seydifa/Brain.git'
REPO_DIR = '/content/Brain'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

Cloning into '/content/Brain'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 32 (delta 2), reused 32 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 26.26 KiB | 26.26 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Brain
Working directory: /content/Brain


## 3 — Link Drive data directory

In [3]:
import os

REPO_DATA = '/content/Brain/data'
DRIVE_DATA = '/content/drive/MyDrive/Brain/data'

# Remove any existing data/ folder in the repo
if os.path.exists(REPO_DATA) and not os.path.islink(REPO_DATA):
    import shutil
    shutil.rmtree(REPO_DATA)
if os.path.islink(REPO_DATA):
    os.remove(REPO_DATA)

os.symlink(DRIVE_DATA, REPO_DATA)
print(f'data/ -> {DRIVE_DATA}')

data/ -> /content/drive/MyDrive/Brain/data


## 4 — Install dependencies

In [4]:
!pip install -q \
    langgraph \
    langgraph-checkpoint-sqlite \
    langchain-google-genai \
    langchain-chroma \
    langchain-ollama \
    ddgs \
    httpx \
    python-dotenv

print('Dependencies installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 81.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 131.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

## 5 — API key
Paste your Gemini API key below (get one free at https://aistudio.google.com/apikey).

In [5]:
import os
from google.colab import userdata

# Option A: store key in Colab Secrets (Colab sidebar > key icon) — recommended
try:
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('Key loaded from Colab Secrets.')
except Exception:
    # Option B: paste directly (less secure)
    os.environ['GOOGLE_API_KEY'] = 'YOUR_GEMINI_API_KEY_HERE'
    print('Key set inline.')

# Write .env so python-dotenv picks it up too
with open('/content/Brain/.env', 'w') as f:
    f.write(f"GOOGLE_API_KEY={os.environ['GOOGLE_API_KEY']}\n")

Key set inline.


## 6 — (Optional) Ollama with local models
Skip this section if you want to use Gemini API.  
On L4 GPU: `llama3.2:3b` responds in ~2 s per call. On T4: ~5 s.

> **No file patching** — the override lives entirely in memory via `sys.modules` injection, so the repo stays clean.

In [ ]:
# Install prerequisites and Ollama
!apt-get install -y -q zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, shutil, os

# Reload PATH so the newly installed binary is found
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')

ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if not os.path.isfile(ollama_bin):
    raise RuntimeError(f'Ollama not found at {ollama_bin} — install may have failed above.')

subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Pull a model — must support tool calling for the ReAct search agent:
# 'llama3.2:3b'    → 2 GB VRAM, fast, full tool support
# 'llama3.1:8b'    → 8 GB VRAM, deeper reasoning, full tool support  ← recommended
# 'mistral:7b'     → 8 GB VRAM, strong reasoning, tool support
OLLAMA_MODEL = 'llama3.1:8b'
!ollama pull {OLLAMA_MODEL}
print(f'Model {OLLAMA_MODEL} ready.')


Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Model llama3.2:3b ready.


In [ ]:
# Override langchain_google_genai with Ollama — no file patching needed
import sys
from types import ModuleType
from langchain_ollama import ChatOllama, OllamaEmbeddings

OLLAMA_MODEL = 'llama3.1:8b'   # larger model — deeper reasoning, full tool support
EMBED_MODEL  = 'nomic-embed-text'

class _OllamaLLM(ChatOllama):
    """Drop-in for ChatGoogleGenerativeAI — ignores Gemini model name."""
    def __init__(self, model=None, temperature=0, **kwargs):
        kwargs.pop('convert_system_message_to_human', None)
        super().__init__(model=OLLAMA_MODEL, temperature=temperature, **kwargs)

class _OllamaEmbeddings(OllamaEmbeddings):
    """Drop-in for GoogleGenerativeAIEmbeddings."""
    def __init__(self, model=None, **kwargs):
        super().__init__(model=EMBED_MODEL, **kwargs)

# Inject before any brain modules are imported
fake = ModuleType('langchain_google_genai')
fake.ChatGoogleGenerativeAI       = _OllamaLLM
fake.GoogleGenerativeAIEmbeddings = _OllamaEmbeddings
sys.modules['langchain_google_genai'] = fake

print(f'Ollama override active')
print(f'  LLM        : {OLLAMA_MODEL}')
print(f'  Embeddings : {EMBED_MODEL}')
print(f'  Repo files : unchanged (no patching)')


Ollama override active
  LLM        : llama3.2:3b
  Embeddings : nomic-embed-text
  Repo files : unchanged (no patching)


## 7 — Load the graph

In [ ]:
import subprocess, sys

# Sync: force-reset to latest GitHub commit (handles diverged history / failed merges)
fetch = subprocess.run(
    ['git', '-C', '/content/Brain', 'fetch', 'origin', 'main'],
    capture_output=True, text=True,
)
reset = subprocess.run(
    ['git', '-C', '/content/Brain', 'reset', '--hard', 'origin/main'],
    capture_output=True, text=True,
)
print('git sync:', reset.stdout.strip() or reset.stderr.strip()[:120])

# Clear .pyc so Python picks up updated bytecode
subprocess.run(['find', '/content/Brain', '-name', '*.pyc', '-delete'], capture_output=True)

# Clear all Brain module caches so updated files are re-imported fresh
brain_mods = [k for k in sys.modules if k.startswith(('core', 'agents', 'memory', 'prompts'))]
for m in brain_mods:
    del sys.modules[m]

sys.path.insert(0, '/content/Brain')

import uuid
from dotenv import load_dotenv
load_dotenv()

from core.graph import get_graph

graph = get_graph()
thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

EMPTY_STATE = {
    'goal': '',
    'messages': [],
    'response': '',
    'status': 'empty',
    'oriented_context': {},
    'reasoning_trace': [],
    'retry_count': 0,
    'search_valid': False,
    'search_feedback': '',
    'qa_draft': '',
    'qa_approved': False,
    'qa_feedback': '',
    'qa_attempts': 0,
    'needs_clarification': False,
    'clarification_reason': '',
    'clarification_questions': [],
}

print('Graph loaded. Ready to ask questions.')


git pull: already up to date
Graph loaded. Ready to ask questions.


In [ ]:
# Verify active runtime configuration
import core.state, subprocess

print(f'MEMORY_SCORE_THRESHOLD : {core.state.MEMORY_SCORE_THRESHOLD}')
print(f'MAX_SEARCH_RETRIES     : {core.state.MAX_SEARCH_RETRIES}')
print(f'MAX_QA_ATTEMPTS        : {core.state.MAX_QA_ATTEMPTS}')

git_log = subprocess.run(
    ['git', '-C', '/content/Brain', 'log', '--oneline', '-4'],
    capture_output=True, text=True,
)
print(f'\nActive commits:\n{git_log.stdout.strip()}')


fetch rc : 0 From https://github.com/Seydifa/Brain
 * branch            main       -> FETCH_HEAD
reset    : HEAD is now at 773c315 fix: graph-load cell uses git fetch + reset --hard to force-sync Colab to latest commits
state.py : MEMORY_SCORE_THRESHOLD = 0.65  # min cosine similarity to count as "known"


In [ ]:
# Verify active model and registered search tools
from agents.search_agent import web_search_tool, academic_search_tool, wikipedia_tool, arxiv_tool, _llm

print('LLM model (Gemini / Ollama override):')
print(' ', getattr(_llm, 'model', '—'))

print('\nSearch tools registered:')
for t in [web_search_tool, academic_search_tool, wikipedia_tool, arxiv_tool]:
    print(f'  ✓ {t.name}')


File says: MEMORY_SCORE_THRESHOLD = 0.50   # min cosine similarity to count as "known"
Module says: MEMORY_SCORE_THRESHOLD = 0.5
Last 3 commits:
 1045299 docs: write full README
be954f8 first commit
9521087 first commit



## 8 — Ask the Brain
Change `goal` and re-run this cell for each question.

In [5]:
import time
from IPython.display import display, Markdown

goal = "What caused World War 2?"  # <-- change this

state = {**EMPTY_STATE, 'goal': goal}

for attempt in range(4):
    try:
        result = graph.invoke(state, config=config)
        break
    except Exception as e:
        if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
            wait = 15 * (attempt + 1)
            print(f'Rate limit — retrying in {wait}s...')
            time.sleep(wait)
        else:
            raise

if result.get('needs_clarification'):
    reason    = result.get('clarification_reason', '')
    questions = result.get('clarification_questions', [])
    why_block = f'\n> **Why:** {reason}\n' if reason else ''
    questions_md = '\n'.join(f'- {q}' for q in questions)
    display(Markdown(f"### 🟡 Clarification needed\n{why_block}\n{questions_md}"))
else:
    display(Markdown(f"### 🟢 Brain Response\n\n{result.get('response', '*(no response)*')}"))

### 🟢 Brain Response

**What caused World War 2?**

The causes of World War II are complex and multifaceted. Some key factors that contributed to the outbreak of the war include:

1. **Rise of Nazi Germany**: Adolf Hitler's aggressive militarism, racist ideology, and desire for territorial expansion created an atmosphere of tension and instability in Europe.
2. **Appeasement Policy**: The policy of appeasement pursued by Britain and France towards Nazi Germany allowed Hitler to pursue his aggressive agenda without facing significant opposition.
3. **Italian Aggression**: Benito Mussolini's fascist regime invaded Ethiopia in 1935, and Japan's military expansion in Asia created a sense of unease among Western powers.
4. **German Rearmament**: Germany's rapid rearmament and remilitarization of the Rhineland in 1936, followed by the annexation of Austria in 1938 (Anschluss) and the Sudetenland in 1938, created a sense of unease among European powers.
5. **Munich Agreement**: The Munich Agreement in 1938 allowed Germany to annex the Sudetenland without facing significant opposition from Britain and France, emboldening Hitler and creating a sense of appeasement that only encouraged further aggression.

These factors, combined with the complex web of alliances and rivalries between European powers, ultimately led to the outbreak of World War II in September 1939.

**Sources:**

[1] (relevance: 0.683)
[2] (relevance: 0.65)
[3] (relevance: 0.643)

## 9 — Debug: inspect reasoning trace

## 8b — Complex test: multi-format stateful thread (4 turns)

Four turns on **CRISPR gene editing** in one thread — exercises every QA format, academic search, and memory continuity:

| Turn | Type | Expected QA format |
|-----:|------|-------------------|
| 1 | `new_topic` | `QUESTION` + academic sources |
| 2 | `follow_up` | `COMPARISON` table |
| 3 | `elaboration` | `QUESTION` follow-up |
| 4 | `follow_up` | `HOW-TO` numbered steps |

In [15]:
import time, uuid
from IPython.display import display, Markdown

# ── Fresh thread so this test runs independently of cell 8 ──────────────────
crispr_thread = str(uuid.uuid4())
crispr_config = {'configurable': {'thread_id': crispr_thread}}

# Four turns — different question types on the same topic
CRISPR_TURNS = [
    # Turn 1 — new topic, QUESTION format, should trigger academic search
    "How does CRISPR-Cas9 gene editing work at the molecular level?",
    # Turn 2 — follow_up, COMPARISON format
    "Compare CRISPR-Cas9 with older gene editing methods: TALEN and zinc finger nucleases.",
    # Turn 3 — elaboration, QUESTION format
    "What are the main ethical concerns and regulatory challenges around human germline editing with CRISPR?",
    # Turn 4 — follow_up, HOW-TO format
    "Give me a step-by-step overview of how a CRISPR-Cas9 experiment is carried out in a lab.",
]

crispr_results = []
t_total = time.time()

for idx, goal in enumerate(CRISPR_TURNS, 1):
    t0 = time.time()
    state = {**EMPTY_STATE, 'goal': goal}

    for attempt in range(4):
        try:
            r = graph.invoke(state, config=crispr_config)
            break
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                wait = 20 * (attempt + 1)
                display(Markdown(f'> ⏳ Rate limit — retrying in {wait}s…'))
                time.sleep(wait)
            else:
                raise

    elapsed   = time.time() - t0
    ctx_r     = r.get('oriented_context', {})
    turn_type = ctx_r.get('turn_type', '?')
    coverage  = ctx_r.get('coverage', '?')
    conf      = ctx_r.get('knowledge_confidence', 0)
    trace     = r.get('reasoning_trace', [])

    if r.get('needs_clarification'):
        outcome = 'CLARIFICATION'
        reason  = r.get('clarification_reason', '')
        why_block = f'\n> **Why:** {reason}\n' if reason else ''
        questions_md = '\n'.join(f'- {q}' for q in r.get('clarification_questions', []))
        body = f'{why_block}\n{questions_md}'
    else:
        outcome = 'ANSWERED'
        body    = r.get('response', '*(no response)*')

    badge = {'ANSWERED': '🟢', 'CLARIFICATION': '🟡'}.get(outcome, '⚪')
    display(Markdown(f"""
---
### Turn {idx}/4 &nbsp; {badge} {outcome}
**Q:** *{goal}*

{body}

> `turn={turn_type}` &nbsp;|&nbsp; `coverage={coverage}` &nbsp;|&nbsp; `conf={conf:.2f}` &nbsp;|&nbsp; `{elapsed:.1f}s`
"""))

    crispr_results.append(dict(
        n=idx, outcome=outcome, turn_type=turn_type,
        coverage=coverage, conf=f'{conf:.2f}', secs=f'{elapsed:.1f}',
    ))

total = time.time() - t_total
rows = '\n'.join(
    f"| {d['n']} | {d['outcome']:<15} | {d['turn_type']:<12} | {d['coverage']:<9} | {d['conf']:>5} | {d['secs']:>5}s |"
    for d in crispr_results
)
display(Markdown(f"""
---
## CRISPR thread summary — {total:.0f}s total

| # | Outcome | Turn type | Coverage | Conf | Time |
|--:|---------|-----------|----------|-----:|------|
{rows}
"""))


---
### Turn 1/4 &nbsp; 🟡 CLARIFICATION
**Q:** *How does CRISPR-Cas9 gene editing work at the molecular level?*


> **Why:** The brain cannot proceed because the goal is too broad and lacks specificity regarding the desired level of detail in the explanation.

- What specific aspect of the CRISPR-Cas9 process would you like to know more about, such as gene targeting or genome editing?
- Is there a particular molecular mechanism or component involved in CRISPR-Cas9 that you'd like me to focus on, like Cas9 enzyme function or guide RNA structure?
- Are you interested in learning about the CRISPR-Cas9 system's interactions with specific types of cells or organisms?

> `turn=follow_up` &nbsp;|&nbsp; `coverage=partial` &nbsp;|&nbsp; `conf=0.50` &nbsp;|&nbsp; `11.7s`



---
### Turn 2/4 &nbsp; 🟢 ANSWERED
**Q:** *Compare CRISPR-Cas9 with older gene editing methods: TALEN and zinc finger nucleases.*

Callback sentence: As we discussed earlier, you were interested in comparing CRISPR-Cas9 with older gene editing methods like TALEN and zinc finger nucleases (ZFNs).

CRISPR-Cas9 has several advantages over its predecessors. **Mechanism:** Unlike ZFNs, which use zinc finger proteins to bind specific DNA sequences, and TALENs, which use transcription activator-like effector nucleases, CRISPR-Cas9 uses the CRISPR-Cas system, involving a guide RNA that targets a specific sequence of DNA. This specificity allows for more precise editing.

**Specificity:** While both ZFNs and TALENs have relatively high specificity rates, CRISPR-Cas9 is generally more accurate, with an error rate significantly lower than its predecessors.

**Efficiency:** CRISPR-Cas9 is also more efficient, requiring lower concentrations to achieve the same level of gene editing as ZFNs and TALENs. This makes it a more accessible technology for researchers studying complex genetic diseases or working with limited resources.

However, both ZFNs and TALENs have their own strengths and are still used in certain applications where CRISPR-Cas9 may not be the best choice. For example, ZFNs can be used to edit genes that are difficult to target with CRISPR-Cas9 due to their location or structure.

In summary, while CRISPR-Cas9 has emerged as a leading tool for genome editing, its advantages come at the cost of increased complexity and potential off-target effects. As research continues to advance, we can expect to see further refinements in this technology.

**Sources:**
[1] (relevance: 0.872)
[2] (relevance: 0.774)
[3] (relevance: 0.731)
[4] (relevance: 0.724)

> `turn=follow_up` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `conf=0.87` &nbsp;|&nbsp; `24.9s`



---
### Turn 3/4 &nbsp; 🟢 ANSWERED
**Q:** *What are the main ethical concerns and regulatory challenges around human germline editing with CRISPR?*

The main ethical concerns surrounding human germline editing with CRISPR involve the potential for unintended consequences, such as introducing new genetic mutations or disrupting the delicate balance of the genome. These concerns are further complicated by issues related to informed consent, particularly in cases where the edited individual may not be aware that they have been genetically modified.

Regulatory challenges arise from the need to balance individual rights with societal interests. Governments and international organizations must navigate complex questions about the ethics of germline editing, including whether it should be permitted for reproductive purposes or only for therapeutic uses. The development of clear guidelines and regulations will be crucial in addressing these concerns and ensuring that CRISPR is used responsibly.

Some key considerations include:

*   **Informed consent:** Ensuring that individuals undergoing germline editing are fully informed about the potential risks and benefits.
*   **Risk assessment:** Conducting thorough risk assessments to identify potential unintended consequences of germline editing.
*   **Regulatory frameworks:** Establishing clear regulatory frameworks to govern the use of CRISPR in humans, including guidelines for informed consent and risk management.

These challenges highlight the need for ongoing dialogue and collaboration among scientists, policymakers, and ethicists to ensure that CRISPR is used in a responsible and ethical manner.

> `turn=follow_up` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `conf=0.79` &nbsp;|&nbsp; `4.7s`



---
### Turn 4/4 &nbsp; 🟢 ANSWERED
**Q:** *Give me a step-by-step overview of how a CRISPR-Cas9 experiment is carried out in a lab.*

As we discussed earlier, you were interested in comparing CRISPR-Cas9 with older gene editing methods like TALEN and zinc finger nucleases (ZFNs).

To carry out a CRISPR-Cas9 experiment in a lab, follow these step-by-step guidelines:

1. **Design and Synthesis of Guide RNAs**: Design and synthesize guide RNAs (gRNAs) that are specific to the target DNA sequence. gRNAs are used to program the Cas9 enzyme to cut at the desired location.
2. **Preparation of the Plasmid**: Prepare a plasmid containing the target gene for transfection into cells.
3. **Transfection and Cell Culture**: Transfect the plasmid into cells and culture them in a growth medium.
4. **CRISPR-Cas9 Reaction**: Mix Cas9 enzyme with the gRNA and the target DNA sequence, and carry out the reaction.
5. **DNA Repair Mechanism**: Activate the cell's DNA repair mechanism to repair any double-strand breaks in the DNA.
6. **Selection of Cells**: Select cells based on their ability to survive the CRISPR-Cas9 reaction and repair the DNA damage.
7. **Verification of Gene Editing**: Verify the edited gene using techniques such as PCR, sequencing, or Western blotting.
8. **Analysis of Results**: Analyze the results to determine the efficiency of the CRISPR-Cas9 system and any off-target effects.

Sources:
[1] (relevance: 0.894)
[2] (relevance: 0.725)
[3] (relevance: 0.717)

> `turn=follow_up` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `conf=0.89` &nbsp;|&nbsp; `14.1s`



---
## CRISPR thread summary — 55s total

| # | Outcome | Turn type | Coverage | Conf | Time |
|--:|---------|-----------|----------|-----:|------|
| 1 | CLARIFICATION   | follow_up    | partial   |  0.50 |  11.7s |
| 2 | ANSWERED        | follow_up    | full      |  0.87 |  24.9s |
| 3 | ANSWERED        | follow_up    | full      |  0.79 |   4.7s |
| 4 | ANSWERED        | follow_up    | full      |  0.89 |  14.1s |


In [16]:
ctx   = result.get('oriented_context', {})
trace = result.get('reasoning_trace', [])

print(f"Turn type  : {ctx.get('turn_type', '?')}")
print(f"Coverage   : {ctx.get('coverage', '?')}")
print(f"Confidence : {ctx.get('knowledge_confidence', 0):.2f}")
print(f"Episode    : {ctx.get('current_episode_id', '?')}")
print()
print('Reasoning trace:')
for i, step in enumerate(trace, 1):
    print(f'  {i}. {step}')

Turn type  : ?
Coverage   : ?
Confidence : 0.00
Episode    : ?

Reasoning trace:


## 10 — View episode history

In [17]:
from memory.episodes import get_recent
import json

episodes = get_recent(10)
for ep in episodes:
    flags = json.loads(ep.get('flags') or '[]')
    print(f"[{ep['id']}]")
    print(f"  Request  : {ep['user_request'][:80]}")
    print(f"  Type     : {ep['turn_type']}")
    print(f"  Flags    : {flags}")
    print(f"  Response : {str(ep.get('chosen_response', ''))[:100]}...")
    print()

[ep_ec650f12_20260413014554]
  Request  : Give me a step-by-step overview of how a CRISPR-Cas9 experiment is carried out i
  Type     : follow_up
  Flags    : ['follow_up', 'search_used']
  Response : As we discussed earlier, you were interested in comparing CRISPR-Cas9 with older gene editing method...

[ep_4cbf335d_20260413014549]
  Request  : What are the main ethical concerns and regulatory challenges around human germli
  Type     : follow_up
  Flags    : ['follow_up', 'search_used']
  Response : The main ethical concerns surrounding human germline editing with CRISPR involve the potential for u...

[ep_da89ab03_20260413014525]
  Request  : Compare CRISPR-Cas9 with older gene editing methods: TALEN and zinc finger nucle
  Type     : follow_up
  Flags    : ['follow_up', 'search_used']
  Response : Callback sentence: As we discussed earlier, you were interested in comparing CRISPR-Cas9 with older ...

[ep_4cedaeb4_20260413014516]
  Request  : How does CRISPR-Cas9 gene editing work a

## 11 — Long stateful conversation (12 turns)
A single thread running through **modern physics**: general relativity → gravitational waves → black holes → dark matter → synthesis.

Tests:
- **Memory continuity** — follow-up classification across many turns
- **Coverage accumulation** — knowledge reuse as the topic deepens
- **Topic pivots** — correct `new_topic` re-classification
- **Synthesis** — final turn draws on everything stored in ChromaDB

In [18]:
import time, textwrap
from IPython.display import display, Markdown

CONVERSATION = [
    # --- Thread 1: General Relativity (turns 1-5) ---
    "Explain the theory of general relativity and what it changed about our understanding of gravity.",
    "How does general relativity differ from Newton's theory of gravity and from special relativity?",
    "What are gravitational waves and how were they first detected?",
    "Who were the key scientists and institutions behind the LIGO gravitational wave detection?",
    "How does GPS rely on both special and general relativity to stay accurate to within metres?",
    # --- Thread 2: Black Holes (turns 6-9) ---
    "What is a black hole and how do they form from dying stars?",
    "What happens at the event horizon of a black hole — can anything escape?",
    "What is Hawking radiation and why does it suggest black holes eventually evaporate?",
    "How are supermassive black holes connected to galaxy formation and active galactic nuclei?",
    # --- Thread 3: Dark Universe (turns 10-11) ---
    "What is dark matter? What observational evidence do we have for its existence?",
    "What is dark energy and how does it explain the accelerating expansion of the universe?",
    # --- Turn 12: Synthesis ---
    "Summarise the key unsolved mysteries in modern physics that we have discussed across all our conversations.",
]

# Reuse the same thread so LangGraph checkpointing + episode diary carry memory
conv_config = {'configurable': {'thread_id': thread_id}}
all_results = []
total_t0 = time.time()

for idx, goal in enumerate(CONVERSATION, 1):
    t0 = time.time()
    state = {**EMPTY_STATE, 'goal': goal}

    for attempt in range(4):
        try:
            r = graph.invoke(state, config=conv_config)
            break
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                wait = 15 * (attempt + 1)
                display(Markdown(f'> ⏳ Rate limit — retrying in {wait}s…'))
                time.sleep(wait)
            else:
                raise

    elapsed   = time.time() - t0
    ctx_r     = r.get('oriented_context', {})
    turn_type = ctx_r.get('turn_type', '?')
    coverage  = ctx_r.get('coverage', '?')
    conf      = ctx_r.get('knowledge_confidence', 0)

    if r.get('needs_clarification'):
        outcome = 'CLARIFICATION'
        reason  = r.get('clarification_reason', '')
        why_block = f'\n> **Why:** {reason}\n' if reason else ''
        questions_md = '\n'.join(f'- {q}' for q in r.get('clarification_questions', []))
        body = f'{why_block}\n{questions_md}'
    else:
        outcome = 'ANSWERED'
        body    = r.get('response', '*(no response)*')

    badge = {'ANSWERED': '🟢', 'CLARIFICATION': '🟡'}.get(outcome, '⚪')
    display(Markdown(f"""
---
### Turn {idx} / {len(CONVERSATION)} &nbsp; {badge} {outcome}

**Q:** *{goal}*

{body}

> `turn={turn_type}` &nbsp;|&nbsp; `coverage={coverage}` &nbsp;|&nbsp; `confidence={conf:.2f}` &nbsp;|&nbsp; `{elapsed:.1f}s`
"""))

    all_results.append(dict(
        q=idx, goal=goal[:60]+'…' if len(goal)>60 else goal,
        outcome=outcome, turn_type=turn_type,
        coverage=coverage, conf=f'{conf:.2f}', time_s=f'{elapsed:.1f}',
    ))

total_elapsed = time.time() - total_t0

rows = '\n'.join(
    f"| {row['q']:>2} | {row['outcome']:<15} | {row['turn_type']:<12} | {row['coverage']:<9} | {row['conf']:>5} | {row['time_s']:>5}s |"
    for row in all_results
)
display(Markdown(f"""
---
## Summary — {len(CONVERSATION)} turns &nbsp; ⏱ {total_elapsed:.0f}s total

| # | Outcome | Turn type | Coverage | Conf | Time |
|--:|---------|-----------|----------|-----:|------|
{rows}
"""))


---
### Turn 1 / 12 &nbsp; 🟢 ANSWERED

**Q:** *Explain the theory of general relativity and what it changed about our understanding of gravity.*

**Theory of General Relativity**

General relativity is a fundamental theory in modern physics that revolutionized our understanding of gravity. Developed by Albert Einstein, it postulates that gravity is not a force between objects, but rather a curvature of spacetime caused by the presence of mass and energy.

According to general relativity, the curvature of spacetime around a massive object such as the Earth causes objects to fall towards the center of the Earth, which we experience as gravity. This theory also predicts phenomena such as gravitational waves, black holes, and the bending of light around massive objects.

**Sources:**

[1] (relevance: 0.877)
[2] (relevance: 0.812)
[3] (relevance: 0.686)

In essence, general relativity changed our understanding of gravity by replacing Newton's law of universal gravitation with a geometric description of gravity, where gravity is an emergent property of spacetime curvature. This theory has had a profound impact on our understanding of the universe, and its predictions have been extensively tested and confirmed through scientific experimentation and observation.

> `turn=new_topic` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.88` &nbsp;|&nbsp; `15.7s`



---
### Turn 2 / 12 &nbsp; 🟢 ANSWERED

**Q:** *How does general relativity differ from Newton's theory of gravity and from special relativity?*

Turn type: new_topic

User goal: How does general relativity differ from Newton's theory of gravity and from special relativity?

Answer:

**Key Differences**

General relativity differs from Newton's theory of gravity and special relativity in several key ways.

Firstly, general relativity **replaces Newton's law of universal gravitation with a geometric description of gravity**, where gravity is an emergent property of spacetime curvature. In contrast, Newton's law describes gravity as a force between objects.

Secondly, general relativity introduces the concept of **spacetime as a dynamic fabric that is curved by mass and energy**. This is in stark contrast to special relativity, which describes spacetime as an absolute background against which physical events occur.

Thirdly, general relativity predicts the existence of **gravitational waves**, which were later detected directly for the first time in 2015. Special relativity does not make this prediction.

Lastly, general relativity predicts the existence of **black holes**, which are regions of spacetime where gravity is so strong that not even light can escape. This is a consequence of the curvature of spacetime around massive objects.

**Sources:**

[1] (relevance: 0.775) Summary: This text provides an overview of Albert Einstein's theory of general relativity, explaining how it revolutionized the understanding of gravity and its effects on spacetime.
[2] (relevance: 0.675) Summary: This text describes the key concepts and implications of Albert Einstein's theory of General Relativity, which revolutionized our understanding of gravity and spacetime.

Note that general relativity also differs from special relativity in other ways, such as its treatment of time dilation and length contraction, but these differences are not explicitly stated here due to space constraints.

> `turn=new_topic` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.78` &nbsp;|&nbsp; `5.7s`



---
### Turn 3 / 12 &nbsp; 🟢 ANSWERED

**Q:** *What are gravitational waves and how were they first detected?*

**Gravitational Waves: A New Window into the Universe**

Gravitational waves are ripples in the fabric of spacetime that are produced by violent cosmic events, such as the collision of two black holes or neutron stars. These waves were predicted by Albert Einstein's theory of general relativity in 1915.

The first direct detection of gravitational waves was announced on February 11, 2016, by the Laser Interferometer Gravitational-Wave Observatory (LIGO). The detection was made using laser interferometry, which involves splitting a laser beam into two perpendicular beams and measuring the tiny changes in distance between the mirrors suspended from the ends of the arms. When a gravitational wave passes through the detector, it causes a minute disturbance in the distance between the mirrors, which is detected as a change in the phase of the laser light.

The signal was consistent with the predicted waveform from binary black hole mergers, and it provided strong evidence for the existence of gravitational waves. The detection was made using data from LIGO's Hanford Observatory in Washington state and its Livingston Observatory in Louisiana.

**Sources:**

1. [1] (relevance: 0.842) - Summary text about the origin and history of gravitational wave detection, specifically referencing LIGO.
2. [2] (relevance: 0.809) - Text describing the detection of gravitational waves using data from LIGO's observatories.
3. [3] (relevance: 0.704) - Text about the observation of gravitational waves from a binary black hole merger by the LIGO Scientific Collaboration in 2016.

Note: I've reformulated the query to make it more specific, as suggested by the web search tool parameters.

> `turn=new_topic` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.84` &nbsp;|&nbsp; `13.5s`



---
### Turn 4 / 12 &nbsp; 🟢 ANSWERED

**Q:** *Who were the key scientists and institutions behind the LIGO gravitational wave detection?*

Turn type: follow_up
Callback sentence: As we discussed earlier, how does general relativity differ from Newton's theory of gravity?

The key scientists behind the LIGO gravitational wave detection were:

1. **Rainer Weiss**: A physicist at MIT who was part of the LIGO Scientific Collaboration and played a crucial role in designing the detector.
2. **Kip Thorne**: A theoretical physicist at Caltech who worked on the development of the LIGO detectors and provided critical insights into the physics of gravitational waves.
3. **Barry Barish**: The director of the LIGO Laboratory, who oversaw the construction and operation of the Hanford and Livingston observatories.

These scientists, along with many others, contributed to the success of the LIGO project, which has revolutionized our understanding of the universe.

Sources:
[1] Weiss, R., et al. (2015). LIGO Scientific Collaboration and Virgo Collaborations. "Observation of Gravitational Waves from a Binary Black Hole Merger." Physical Review Letters, 116(11), 061102.
[2] Abbott, B. P., et al. (2016). LIGO Scientific Collaboration and Virgo Collaborations. "GW150914: The First Observation of Gravitational Waves from a Binary Black Hole Merger." Astrophysical Journal Letters, 818(L1), L1-L9.

> `turn=follow_up` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.77` &nbsp;|&nbsp; `4.8s`



---
### Turn 5 / 12 &nbsp; 🟢 ANSWERED

**Q:** *How does GPS rely on both special and general relativity to stay accurate to within metres?*

Turn type: follow_up

Callback sentence: As I mentioned earlier, general relativity differs from Newton's theory of gravity in its description of gravity as an emergent property of spacetime curvature.

The GPS system relies on both special and general relativity to stay accurate to within meters due to several key effects. Firstly, **time dilation** causes the satellite clocks to run faster than Earth-based clocks by about 38 microseconds per day, resulting in a positional error of around 10 km per day. This effect is a direct consequence of special relativity.

Secondly, **gravitational redshift** further reduces the accuracy of the satellite clocks due to their weaker gravitational field. The Sagnac effect, which occurs as the GPS satellites move relative to an observer on Earth, also results in a phase shift between the signals transmitted by the satellite and received by the receiver.

To account for these relativistic effects, the GPS system uses a combination of advanced technologies, including high-precision atomic clocks that are synchronized with Earth-based clocks. These clocks ensure accurate timekeeping, which is essential for providing location and time information to within meters and seconds, respectively.

Sources:
[1] (relevance: 0.709)
Summary: This text explains how General Relativity affects the accuracy of GPS systems, specifically through three key effects: time dilation, gravitational redshift, and the Sagnac effect.
Content: ...

[2] (relevance: 0.708)
Summary: This text discusses how the Global Positioning System (GPS) accounts for relativistic effects, including phase shifts and synchronization issues caused by special and general relativity, through advanced technologies such as high-precision atomic clocks and relativistic corrections.
Content: ...

[3] (relevance: 0.675)
Summary: This text provides an overview of Albert Einstein's theory of general relativity, explaining how it revolutionized the understanding of gravity and its effects on spacetime.
Content: ...
 
[4] (relevance: 0.67)
Summary: This text describes how the GPS system uses orbit determination algorithms that account for relativistic effects to achieve high accuracy in predicting satellite positions and velocities.
Content: ...

> `turn=follow_up` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.71` &nbsp;|&nbsp; `128.9s`


...


---
### Turn 9 / 12 &nbsp; 🟢 ANSWERED

**Q:** *How are supermassive black holes connected to galaxy formation and active galactic nuclei?*

Turn type: follow-up

Callback sentence: As I mentioned earlier, general relativity differs from Newton's theory of gravity in its description of gravity as an emergent property of spacetime curvature.

The connection between supermassive black holes and galaxy formation is a complex one. Supermassive black holes are believed to reside at the centers of most galaxies, including our own Milky Way. These black holes are thought to play a crucial role in the growth and evolution of galaxies.

[1] (relevance: 0.681)
Summary: This text describes the process by which a massive star's collapse leads to the formation of a black hole and its characteristics, specifically the event horizon and singularity.
Content: ...the core of the star eventually reaches a point where its gravity becomes so strong that not even light can escape.

The presence of a supermassive black hole at the center of a galaxy can have several effects on the surrounding environment. For one, it can regulate the growth of stars within the galaxy by controlling the amount of gas and dust available for star formation. This is because the strong gravity of the black hole can heat up the surrounding gas, preventing it from cooling and collapsing into new stars.

[2] (relevance: 0.679)
Summary: This text describes the formation and characteristics of a black hole, including its accretion disk, singularity, and event horizon.
Content: ...the core of the star eventually reaches a point where its gravity becomes so strong that not even light can escape.

Furthermore, supermassive black holes are thought to be responsible for the growth of galaxies through a process known as active galactic nucleus (AGN) activity. This occurs when hot gas from the galaxy's disk is accreted onto the black hole, causing it to grow in mass and emit intense radiation across the entire electromagnetic spectrum.

[3] (relevance: 0.671)
Summary: This text explains how a black hole is formed when a massive star collapses at the end of its life cycle, resulting in an incredibly strong gravitational pull that traps everything, including light.
Content: ...the core of the star eventually reaches a point where its gravity becomes so strong that not even light can escape.

In summary, supermassive black holes play a crucial role in the growth and evolution of galaxies through their regulation of star formation and AGN activity. The connection between these two phenomena is still an active area of research, but it is clear that supermassive black holes are integral to understanding the history and development of our universe.

Sources:
[1], [2], and [3] provide a comprehensive overview of the process by which supermassive black holes form and their characteristics.

> `turn=follow_up` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.68` &nbsp;|&nbsp; `8.1s`



---
### Turn 10 / 12 &nbsp; 🟢 ANSWERED

**Q:** *What is dark matter? What observational evidence do we have for its existence?*

Dark matter is a hypothetical form of matter that is thought to make up approximately 95% of the universe, yet its exact nature remains unknown.

The existence of dark matter was first proposed by Swiss astrophysicist Fritz Zwicky in the 1930s, based on his observations of galaxy clusters. He realized that the galaxies within these clusters were moving at much higher velocities than expected, suggesting that there was a large amount of unseen mass holding them together.

Since then, a wealth of observational evidence has accumulated to support the existence of dark matter. Some of the key lines of evidence include:

1. **Galactic Rotation Curves**: The rotation curves of galaxies are the rate at which stars and gas orbit around the center of the galaxy. These curves are typically flat or even rise slightly towards the edges of the galaxy, indicating that there is a large amount of unseen mass.
2. **Galaxy Clusters**: The distribution of galaxy clusters and the hot gas between them indicate that there is a large amount of unseen mass holding these systems together.
3. **Large-Scale Structure of the Universe**: The universe's large-scale structure, including the distribution of galaxies and galaxy clusters, can be explained by the presence of dark matter.

Sources:

[1] (relevance: 0.684)
Summary: This text discusses five unsolved mysteries in modern physics, including phenomena such as dark matter and dark energy, that continue to puzzle scientists despite making up a significant portion of the universe.
Content: Based on my search, here are some of the key unsolved mysteries in modern physics that we haven't discussed yet:

1. **The Nature of Dark Matter and Dark Energy**: These two phenomena make up approximately 95% of the universe, but their exact nature remains unknown.

[2] (relevance: 0.651)
Summary: This text describes the detection of gravitational waves using data from LIGO's observatories, which provided strong evidence for their existence and opened up new avenues for studying cosmic phenomena.
Content: a gravitational wave passes through the detector, it causes a minute disturbance in the distance between the mirrors, which is detected as a change in the phase of the laser light.

The detection was made using data from LIGO's Hanford Observatory in Washington state and its Livingston Observatory in Louisiana. The signal was consistent with the predicted waveform from binary black hole mergers, and it provided strong evidence for the existence of gravitational waves.

The discovery of gravitational waves has opened up a new window into the universe, allowing us to study cosmic phenomena in ways that were previously impossible. It has also confirmed many of the predictions made by Einstein's theory of general relativity and has paved the way for further research into the properties of gravity.

> `turn=new_topic` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.68` &nbsp;|&nbsp; `7.9s`



---
### Turn 11 / 12 &nbsp; 🟢 ANSWERED

**Q:** *What is dark energy and how does it explain the accelerating expansion of the universe?*

Dark energy is a mysterious component that is thought to be responsible for the accelerating expansion of the universe.

The accelerating expansion of the universe was first observed in the late 1990s by a team of scientists led by Saul Perlmutter, Adam Riess, and Brian Schmidt. They found that the expansion of the universe was not slowing down, as one would expect due to the gravitational pull of matter, but was instead speeding up.

[1] This phenomenon is attributed to dark energy, which is thought to make up approximately 68% of the universe's total energy density. The exact nature of dark energy remains unknown, and it is believed to be a property of space itself rather than a type of matter or radiation.

One way to understand dark energy is to consider its effects on the expansion of the universe. According to the Friedmann-Lemaître-Robertson-Walker (FLRW) model, which describes the evolution of the universe, the expansion of the universe should slow down over time due to the gravitational pull of matter. However, observations suggest that this is not happening, and instead, the expansion is accelerating.

This acceleration is thought to be caused by dark energy, which is pushing matter apart and causing the distance between galaxies to increase. The exact mechanism behind this effect is still not well understood, but it is believed to be related to the properties of dark energy itself.

Sources:
[1] (relevance: 0.696)
[2] (not directly relevant, but provides context on the FLRW model)

> `turn=follow_up` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.70` &nbsp;|&nbsp; `5.1s`



---
### Turn 12 / 12 &nbsp; 🟢 ANSWERED

**Q:** *Summarise the key unsolved mysteries in modern physics that we have discussed across all our conversations.*

Turn type: new_topic

User goal: Summarise the key unsolved mysteries in modern physics that we have discussed across all our conversations.

Context:
Relevant knowledge chunks:

1. **The Nature of Dark Matter and Dark Energy**: These two phenomena make up approximately 95% of the universe, but their exact nature remains unknown.
2. **The Hierarchy Problem**: The difference between the gravitational force and the electromagnetic force is difficult to explain using current theories.
3. **The Black Hole Information Paradox**: What happens to information contained in matter that falls into a black hole?
4. **The Quantum Gravity Problem**: A complete theory of quantum gravity that merges general relativity and quantum mechanics remains elusive.
5. **The Origin of the Universe's Matter-Antimatter Asymmetry**: Why is the universe dominated by matter, rather than antimatter?

To summarize the key unsolved mysteries in modern physics that we've discussed across our conversations:

We've explored several fundamental questions in modern physics, including dark matter and dark energy, which together comprise approximately 95% of the universe. The hierarchy problem, black hole information paradox, quantum gravity problem, and origin of the universe's matter-antimatter asymmetry remain open questions that continue to puzzle scientists.

Sources:
[1] Modern Physics Unsolved Mysteries
[2] General Relativity: A Cornerstone of Modern Physics
[3] Antimatter: An Open Question in Modern Physics

> `turn=new_topic` &nbsp;|&nbsp; `coverage=full` &nbsp;|&nbsp; `confidence=0.81` &nbsp;|&nbsp; `4.8s`



---
## Summary — 12 turns &nbsp; ⏱ 226s total

| # | Outcome | Turn type | Coverage | Conf | Time |
|--:|---------|-----------|----------|-----:|------|
|  1 | ANSWERED        | new_topic    | full      |  0.88 |  15.7s |
|  2 | ANSWERED        | new_topic    | full      |  0.78 |   5.7s |
|  3 | ANSWERED        | new_topic    | full      |  0.84 |  13.5s |
|  4 | ANSWERED        | follow_up    | full      |  0.77 |   4.8s |
|  5 | ANSWERED        | follow_up    | full      |  0.71 | 128.9s |
|  6 | ANSWERED        | follow_up    | full      |  0.82 |  21.8s |
|  7 | ANSWERED        | new_topic    | full      |  0.71 |   3.8s |
|  8 | ANSWERED        | new_topic    | full      |  0.72 |   6.2s |
|  9 | ANSWERED        | follow_up    | full      |  0.68 |   8.1s |
| 10 | ANSWERED        | new_topic    | full      |  0.68 |   7.9s |
| 11 | ANSWERED        | follow_up    | full      |  0.70 |   5.1s |
| 12 | ANSWERED        | new_topic    | full      |  0.81 |   4.8s |


In [ ]:
import time, uuid
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Fresh thread (isolated from every previous test) ─────────────────────────
q30_thread = str(uuid.uuid4())
q30_config = {'configurable': {'thread_id': q30_thread}}

# ── 30 questions — topic arc ──────────────────────────────────────────────────
# Q01-07  Quantum computing foundations
# Q08-12  Hardware & engineering
# Q13-16  Advanced topics
# Q17     ⚡ TOPIC CHANGE — interpretations of quantum mechanics
# Q18-23  QM foundations & philosophy
# Q24     🔄 COMEBACK — connect both threads
# Q25-26  Quantum cryptography
# Q27-29  Quantum sensing & networks
# Q30     Synthesis
Q30_TURNS = [
    # ── Foundations (Q1-7) ──────────────────────────────────────────────────
    "What is a qubit and how does superposition make it fundamentally different from a classical bit?",
    "Explain quantum entanglement — what it is physically, how it is created in the lab, and how it enables correlations that Einstein called 'spooky action at a distance'.",
    "What is quantum decoherence, why does it destroy quantum states, and what timescales are relevant in real superconducting hardware?",
    "Walk me through how quantum gates work — explain the Hadamard, CNOT, and Toffoli gates and their effect on qubit states.",
    "How does Shor's algorithm factor large integers exponentially faster than classical computers? Explain the quantum Fourier transform's role.",
    "Explain Grover's algorithm — what quadratic speedup does it achieve and for what class of problems is it applicable?",
    "What is quantum error correction? State the quantum error-correction conditions and explain why classical error-correction ideas cannot be directly applied.",
    # ── Hardware & engineering (Q8-12) ──────────────────────────────────────
    "Explain the surface code in detail: logical qubits, stabilizer operators, syndrome measurement, and how errors are detected and corrected.",
    "What is the fault-tolerance threshold theorem? What physical gate error rate do current superconducting processors achieve versus the theoretical threshold?",
    "Compare superconducting qubits (IBM, Google), trapped ions (IonQ, Quantinuum), and photonic approaches — tradeoffs in coherence time, gate fidelity, and scalability.",
    "What is the NISQ era? What algorithms can run on today's noisy 100-1000 qubit devices without full error correction?",
    "Critically assess the quantum supremacy claim by Google's Sycamore (2019) and the subsequent counter-arguments from IBM and classical simulation results.",
    # ── Advanced topics (Q13-16) ─────────────────────────────────────────────
    "What are topological qubits? Explain non-Abelian anyons, braiding operations, and why topological protection is theoretically superior for fault tolerance.",
    "How does the variational quantum eigensolver (VQE) work, what Hamiltonian problems does it target, and what are its practical limitations in the NISQ era?",
    "What is quantum volume as a hardware benchmark, and how does it capture holistic device performance better than raw qubit count?",
    "What is the current (2025) state of quantum computing — milestones reached, leading hardware roadmaps, and a realistic timeline to fault-tolerant quantum computing?",
    # ── ⚡ TOPIC CHANGE: Foundations of quantum mechanics (Q17-23) ───────────
    "Setting quantum computers aside — what exactly is the measurement problem in quantum mechanics, and why has it resisted resolution since 1927?",
    "Compare the Copenhagen interpretation, Everett's Many-Worlds, and de Broglie–Bohm pilot-wave theory. What philosophical and empirical grounds distinguish them?",
    "How does decoherence attempt to explain the apparent collapse of the wave function and the emergence of the classical world? Why do critics say it does not fully solve the measurement problem?",
    "What is Carlo Rovelli's relational interpretation of quantum mechanics, and how does QBism (quantum Bayesianism) differ in its treatment of the quantum state?",
    "What exactly does Bell's theorem prove? How do the loophole-free Bell tests (Aspect, Zeilinger, Hensen) constrain local hidden-variable theories?",
    "Is the wave function a real physical object (ψ-ontic) or merely an agent's knowledge state (ψ-epistemic)? What does the Pusey-Barrett-Rudolph theorem imply?",
    "Could quantum mechanics be an emergent effective theory from a deeper deterministic substrate — analogous to thermodynamics emerging from statistical mechanics?",
    # ── 🔄 COMEBACK: Connect both threads (Q24-26) ───────────────────────────
    "Returning to quantum computing through what we just discussed — how does the measurement problem and decoherence theory bear on quantum error correction? Does the interpretation you adopt affect how you design a quantum computer?",
    "What is the quantum computing threat to cryptography? When could a fault-tolerant quantum computer break RSA-2048, and what physical resources — logical qubits, T-gates — would that require?",
    "Which post-quantum cryptography algorithms has NIST standardised (ML-KEM, ML-DSA, SLH-DSA)? How does lattice hardness (LWE/NTRU) resist quantum attacks including Grover's algorithm?",
    # ── Sensing & networks (Q27-29) ─────────────────────────────────────────
    "What is quantum sensing? How do atomic clocks, atom-interferometer gravimeters, and nitrogen-vacancy magnetometers exploit quantum coherence for sensitivity beyond the standard quantum limit?",
    "How does quantum key distribution work? Explain the BB84 protocol, the role of the no-cloning theorem, and why QKD provides information-theoretic rather than computational security.",
    "What is a quantum repeater and why is it needed for a global quantum internet? What physical implementations exist and what are the main engineering challenges?",
    # ── Synthesis (Q30) ─────────────────────────────────────────────────────
    "Synthesise the full arc of our conversation: from qubits and decoherence to error correction, interpretations of quantum mechanics, connecting foundations to engineering, quantum cryptography, and quantum sensing. What are the three deepest open questions at the intersection of quantum foundations and quantum technology?",
]
assert len(Q30_TURNS) == 30, f"Expected 30 questions, got {len(Q30_TURNS)}"

# Section markers used for the confidence trend chart
SECTIONS_Q30 = {
    1:  "Foundations",
    8:  "Hardware",
    13: "Advanced",
    17: "Topic change",
    24: "Comeback",
    27: "Sensing/Networks",
    30: "Synthesis",
}

# Section style config for chart (key = first question in section)
_SECT_STYLE = {
    1:  dict(color="steelblue",  lw=1.0, ls="--"),
    8:  dict(color="slategray",  lw=1.0, ls="--"),
    13: dict(color="slategray",  lw=1.0, ls="--"),
    17: dict(color="darkorange", lw=2.2, ls="-"),
    24: dict(color="purple",     lw=2.2, ls="-"),
    27: dict(color="slategray",  lw=1.0, ls="--"),
    30: dict(color="slategray",  lw=1.0, ls="--"),
}

# ── Run 30-turn conversation ──────────────────────────────────────────────────
q30_results = []
t0_total = time.time()

for idx, goal in enumerate(Q30_TURNS, 1):
    t0 = time.time()
    state = {**EMPTY_STATE, 'goal': goal}

    r = {
        'response': 'Error: max retries exceeded',
        'needs_clarification': False,
        'oriented_context': {},
        'reasoning_trace': [],
    }
    for attempt in range(4):
        try:
            r = graph.invoke(state, config=q30_config)
            break
        except Exception as exc:
            if '429' in str(exc) or 'RESOURCE_EXHAUSTED' in str(exc):
                wait = 20 * (attempt + 1)
                display(Markdown(f'> ⏳ Rate limit on Q{idx} — retrying in {wait}s…'))
                time.sleep(wait)
            elif attempt < 3:
                time.sleep(5)
            else:
                display(Markdown(f'> ❌ Q{idx} failed after 4 attempts: `{str(exc)[:120]}`'))

    elapsed   = time.time() - t0
    ctx_r     = r.get('oriented_context', {})
    turn_type = ctx_r.get('turn_type', '?')
    coverage  = ctx_r.get('coverage', '?')
    conf      = float(ctx_r.get('knowledge_confidence', 0.0))
    trace_len = len(r.get('reasoning_trace', []))

    if r.get('needs_clarification'):
        outcome   = 'CLARIFICATION'
        reason    = r.get('clarification_reason', '')
        why_block = f'\n> **Why:** {reason}\n' if reason else ''
        qs_md     = '\n'.join(f'- {q}' for q in r.get('clarification_questions', []))
        body      = f'{why_block}\n{qs_md}'
    else:
        outcome = 'ANSWERED'
        body    = r.get('response', '*(no response)*')

    # Determine section label for this turn
    section = next(
        SECTIONS_Q30[s]
        for s in sorted(SECTIONS_Q30.keys(), reverse=True)
        if idx >= s
    )

    badge = {'ANSWERED': '🟢', 'CLARIFICATION': '🟡'}.get(outcome, '⚪')
    display(Markdown(f"""---
### Q{idx}/30 &nbsp; {badge} {outcome} &nbsp; · &nbsp; *{section}*
**Q:** *{goal}*
{body}

> `turn_type={turn_type}` &nbsp;|&nbsp; `coverage={coverage}` &nbsp;|&nbsp; `conf={conf:.2f}` &nbsp;|&nbsp; `steps={trace_len}` &nbsp;|&nbsp; `{elapsed:.1f}s`"""))

    q30_results.append(dict(
        n=idx, outcome=outcome, turn_type=turn_type,
        coverage=coverage, conf=conf, secs=elapsed,
    ))

elapsed_total = time.time() - t0_total

# ── Summary table ─────────────────────────────────────────────────────────────
rows = '\n'.join(
    f"| {d['n']:>2} | {'🟢' if d['outcome'] == 'ANSWERED' else '🟡'} {d['outcome']:<14} "
    f"| {d['turn_type']:<12} | {d['coverage']:<9} | {d['conf']:>5.2f} | {d['secs']:>5.1f}s |"
    for d in q30_results
)
display(Markdown(f"""---
## 30-turn summary &nbsp; ⏱ {elapsed_total / 60:.1f} min total

| # | Outcome | Turn type | Coverage | Conf | Time |
|--:|---------|-----------|----------|-----:|------|
{rows}
"""))

# ── Confidence trend chart ─────────────────────────────────────────────────────
turns_x = [d['n']    for d in q30_results]
confs_y = [d['conf'] for d in q30_results]

# 4-turn rolling average (point i = mean of last min(4, i) turns)
window  = 4
rolling = [
    sum(confs_y[max(0, i - window + 1):i + 1]) / min(window, i + 1)
    for i in range(len(confs_y))
]

outcome_colors = [
    '#2ecc71' if d['outcome'] == 'ANSWERED' else '#e67e22'
    for d in q30_results
]
type_palette = {
    'new_topic':    '#e74c3c',
    'follow_up':    '#2ecc71',
    'elaboration':  '#3498db',
    'clarification':'#9b59b6',
    'correction':   '#f39c12',
    '?':            '#95a5a6',
}

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(17, 9),
    gridspec_kw={'height_ratios': [3, 1]},
)

# ── Top panel: confidence trend ───────────────────────────────────────────────
ax1.fill_between(turns_x, confs_y, alpha=0.10, color='steelblue')
ax1.plot(turns_x, confs_y, color='steelblue',  linewidth=1.5, alpha=0.65,
         label='Per-turn confidence')
ax1.plot(turns_x, rolling, color='royalblue',  linewidth=2.5,
         label=f'{window}-turn rolling avg (knowledge trend)')
ax1.scatter(turns_x, confs_y, c=outcome_colors, s=70, zorder=6,
            edgecolors='white', linewidths=0.5)

# Threshold line
ax1.axhline(0.65, color='gray', linestyle=':', linewidth=1.2, alpha=0.8,
            label='Memory threshold (0.65)')

# Section dividers + labels
for q_num, label in SECTIONS_Q30.items():
    st = _SECT_STYLE.get(q_num, dict(color='gray', lw=1, ls='--'))
    if q_num > 1:
        ax1.axvline(q_num - 0.5, color=st['color'], linestyle=st['ls'],
                    linewidth=st['lw'], alpha=0.75)
    ax1.text(q_num + 0.1, 1.05, label, fontsize=7.5,
             color=st['color'], rotation=40, va='bottom', ha='left')

ax1.set_xlim(0.5, 30.7)
ax1.set_ylim(0, 1.30)
ax1.set_xticks(range(1, 31))
ax1.set_xlabel('Turn #', fontsize=11)
ax1.set_ylabel('Knowledge confidence', fontsize=11)
ax1.set_title(
    'Knowledge Confidence Trend — 30 turns\n'
    'Quantum Computing (Q1-16)  →  QM Interpretations (Q17-23)  →  Comeback (Q24-26)  →  Synthesis (Q30)',
    fontsize=11, fontweight='bold',
)
answered_patch  = mpatches.Patch(color='#2ecc71', label='🟢 ANSWERED')
clarif_patch    = mpatches.Patch(color='#e67e22', label='🟡 CLARIFICATION')
ax1.legend(
    handles=[answered_patch, clarif_patch] + ax1.get_lines()[1:4],
    loc='upper left', fontsize=9, ncol=2,
)
ax1.grid(True, alpha=0.18)

# ── Bottom panel: turn-type bars ──────────────────────────────────────────────
for d in q30_results:
    ax2.bar(d['n'], 1, color=type_palette.get(d['turn_type'], '#95a5a6'),
            alpha=0.85, width=0.8)

for q_num in SECTIONS_Q30:
    if q_num > 1:
        st = _SECT_STYLE.get(q_num, dict(color='gray', lw=1, ls='--'))
        ax2.axvline(q_num - 0.5, color=st['color'], linestyle=st['ls'],
                    linewidth=st['lw'], alpha=0.70)

ax2.set_xlim(0.5, 30.7)
ax2.set_ylim(0, 1.1)
ax2.set_xticks(range(1, 31))
ax2.set_xlabel('Turn #', fontsize=11)
ax2.set_yticks([])
ax2.set_title('Turn Classification (memory continuity)', fontsize=10)
legend_patches = [
    mpatches.Patch(color=c, label=t)
    for t, c in type_palette.items() if t != '?'
]
ax2.legend(handles=legend_patches, loc='upper right', fontsize=8, ncol=5)
ax2.grid(True, alpha=0.15, axis='x')

plt.tight_layout(h_pad=2.5)

chart_path = '/content/Brain/quantum_confidence_trend.png'
plt.savefig(chart_path, dpi=130, bbox_inches='tight')
plt.show()
display(Markdown(f'Chart saved → `{chart_path}`'))
